# 25 - Full Carvana inventory: collection and coverage

**Question:** Did the expanded collector finish its declared inventory, and where are the gaps?

**Inputs:** retained full-inventory reports and an optional reviewed export. **Outputs:** read-only coverage tables. Run All never collects or writes. This workflow is separate from the frozen 101-query experiment. Read `vehicle/docs/full_inventory.md` for operation and `full_inventory_goal.md` for acceptance criteria.

A VIN identifies a vehicle. A query selects a recorded make, model and/or year context at a ZIP. An observation is a vehicle on a retained search page at its actual observation time. A six-hour sweep is not a simultaneous snapshot. Missing vehicles are not automatically sales.


In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
REPO = next(p for p in [here, *here.parents] if (p / 'vehicle/config/carvana_full_inventory.json').is_file())
CONFIG_NAMES = ['carvana_full_inventory.json', 'carvana_full_inventory_years.json']
configs = {name: json.loads((REPO / 'vehicle/config' / name).read_text()) for name in CONFIG_NAMES}
config = configs[CONFIG_NAMES[-1]]
# Include the reviewed execution roots while source-bound captures remain in worktrees.
root_values = [item['capture_root'] for item in configs.values()] + config['related_capture_roots']
CAPTURE_ROOTS = sorted({(REPO / 'vehicle' / value).resolve() for value in root_values}, key=str)
EXPORT = None  # Optional: Path to one existing replay/export folder.
print('Request ceiling:', config['max_requests'], '| Maximum hours:', config['max_seconds'] / 3600)
print('Primary ZIP:', config['primary_zip'], '| Validation ZIPs:', config['validation_zips'])


## Collection register

These are operational summaries, not verified sales or proof of national coverage. `declared_collection_complete` means discovery, primary queries and declared ZIP checks finished. `primary_queries_complete` covers the primary leaves alone. `primary_scope_reconciled` additionally requires no duplicate primary membership and zero opening/closing count residual. Geographic checks cover a limited rotating sample. Missing dates are not zero-inventory dates. The original all-year and prospective year-based attempts remain distinct rows; this register does not replace a predeclared seven-date trial denominator.


The register reads the capture roots explicitly listed in the reviewed configurations, including the original execution worktree. Each report keeps its source path and configuration hash; it does not combine attempts or copy their evidence.


In [ ]:
records = []
for capture_root in CAPTURE_ROOTS:
    for path in sorted(capture_root.glob('*/catalog_report.json')):
        report = json.loads(path.read_text(encoding='utf-8'))
        record = {key: report.get(key) for key in [
            'cycle_date', 'partition_strategy', 'status', 'requests', 'discovery_complete', 'primary_queries_complete',
            'declared_collection_complete', 'primary_scope_reconciled', 'primary_observed_vins', 'opening_count_residual',
            'closing_count_residual', 'geographic_membership_stable', 'started_at', 'ended_at']}
        if record['partition_strategy'] is None and report.get('format') == 'carvana-full-inventory-run-v1':
            record['partition_strategy'] = 'all_year_models'
        record.update(capture_root=str(capture_root), config_sha256=report.get('config_sha256'), report_path=str(path))
        records.append(record)
collection_register = pd.DataFrame(records)
if collection_register.empty:
    print('No full-inventory baseline retained yet. Updated code and budget are not collected evidence.')
else:
    display(collection_register)


## Inspect one replayed export

Set `EXPORT` to an existing export when ready. Its manifest binds source and output bytes. Discovery samples and geographic checks are excluded from primary inventory, except complete make probes. A blocked replay requires evidence review, not replacement with a fresh attempt.


In [ ]:
if EXPORT is not None:
    EXPORT = Path(EXPORT)
    manifest = json.loads((EXPORT / 'manifest.json').read_text(encoding='utf-8'))
    for path, expected in manifest['sources'].items():
        assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == expected, f'Changed source: {path}'
    for name, expected in manifest['outputs'].items():
        assert hashlib.sha256((EXPORT / name).read_bytes()).hexdigest() == expected, f'Changed export: {name}'
    coverage = pd.read_csv(EXPORT / 'coverage.csv')
    display(coverage)
    display(coverage.loc[~coverage['query_complete'].fillna(False)])
    display(pd.read_json(EXPORT / 'geographic_checks.json'))
    display(pd.read_csv(EXPORT / 'make_reconciliation.csv'))
    if (EXPORT / 'year_reconciliation.csv').is_file():
        display(pd.read_csv(EXPORT / 'year_reconciliation.csv'))
        display(pd.DataFrame(json.loads((EXPORT / 'native_zero_categories.json').read_text(encoding='utf-8'))))
        print('Native zero categories are source counts, not fabricated inventory-query reports. Missing years remain explicit uncertainty.')
    print('Incomplete make counts are observed lower bounds, not zero inventory or evidence of absence.')
else:
    print('No export selected; nothing has been imported, refreshed or collected.')


## Move from coverage to sales research

Compare each VIN with all retained positive observations, preserving first/last observed times and source scope. Only comparable complete coverage can support absence. New category coverage can discover old inventory: first observed does not mean newly listed.

Continue to Notebook 20 for compatible inventory/price comparisons and Notebook 24 for native-status validation. Keep pending, absent, native Sold and transaction-confirmed sale separate. The existing small validation study cannot establish a companywide sales conversion rate.


## Compare with explicitly selected history

Set an aware `HISTORY_AS_OF` cutoff and select retained exports or legacy cycle/database pairs. Each membership keeps its capture, context and clocks. Partial captures can establish a sighting, never an absence. Catalog analysis is available at export publication, while first/last sightings retain actual observation times. The tables do not estimate sales.


In [ ]:
HISTORY_AS_OF = None  # Example: an explicit UTC timestamp after the selected exports were published.
CATALOG_EXPORTS = []  # Explicit existing analysis directories; include EXPORT if desired.
LEGACY_SOURCES = []  # Each item: {'cycle_paths': [Path(...)], 'database': Path(...) or None}.
if HISTORY_AS_OF is not None:
    import sys
    sys.path.insert(0, str(REPO / 'vehicle/src'))
    from vehicle_tracker.catalog_history import observed_catalog_history
    vin_history, first_seen_cohorts, source_memberships = observed_catalog_history(
        catalog_exports=CATALOG_EXPORTS, legacy_sources=LEGACY_SOURCES, as_of=HISTORY_AS_OF)
    display(vin_history)
    display(first_seen_cohorts)
    display(source_memberships)
else:
    print('No historical inputs or cutoff selected; existing evidence remains unchanged.')
